# VGG 최종 개선 파이프라인

최신 5-fold 운전자 CV + 전체/크롭 확률 결합 + 얼굴 조건부 보정 + 클래스별 확률 보정입니다.
원본 State Farm 데이터와 ImageNet 가중치만으로 처음부터 실행할 수 있습니다.
이 노트북은 이전 비공개 Kaggle 노트북 출력에 의존하지 않습니다.

Kaggle에서는 공식 대회 데이터, GPU와 Internet을 켜고 위에서부터 실행합니다.
입력 루트는 STATEFARM_INPUT_ROOT, 출력 루트는 STATEFARM_OUTPUT_ROOT로 변경할 수 있습니다.
최종 파일은 `/kaggle/working/vgg_final/final/submission.csv`입니다.
각 fold 고정 5에폭이며 10개 전체/크롭 모델, 5개 얼굴 OOF 모델, 1개 전체 얼굴 모델을 학습합니다.
검증 운전자는 해당 base 모델에서 제외됩니다. 보정은 완전한 nested CV가 아닌 개발용 교차 검증입니다.
이 고정 설정은 이전 개발 실험의 영향을 받았습니다.

현재 기록된 26명 OOF: Log Loss 0.25035 / 정확도 91.80%.
최신 CSV의 Kaggle 점수는 실제 제출 전에는 알 수 없습니다. 이 노트북은 자동 제출하지 않습니다.


## 1. 전체 이미지 VGG 5-fold + 얼굴 OOF

In [ ]:
from pathlib import Path
import os
INPUT_ROOT=Path(os.environ.get('STATEFARM_INPUT_ROOT','/kaggle/input'))
_default_output='/kaggle/working/vgg_final' if Path('/kaggle/working').is_dir() else './vgg_final_outputs'
WORK_ROOT=Path(os.environ.get('STATEFARM_OUTPUT_ROOT',_default_output))
WORK_ROOT.mkdir(parents=True,exist_ok=True)
CV_CONFIG = {'driver_folds': {'p002': 0, 'p026': 0, 'p035': 0, 'p041': 0, 'p050': 0, 'p075': 0, 'p081': 1, 'p051': 1, 'p015': 1, 'p024': 1, 'p012': 1, 'p014': 2, 'p056': 2, 'p052': 2, 'p016': 2, 'p061': 2, 'p064': 3, 'p047': 3, 'p072': 3, 'p022': 3, 'p045': 3, 'p066': 4, 'p039': 4, 'p049': 4, 'p021': 4, 'p042': 4}, 'metadata_sha256': '891ca8e2bfd7a3522ac2098a1bc93825c581d07104d211c329de0b16c83ca5ee', 'fixed_epochs': 5, 'schedule_epochs': 9, 'fixed_variant': 'EMA .995; BN refreshed on fit drivers only', 'face_fixed_epochs': 2, 'family_comparison': 'full image vs fixed driver crop; same VGG16-BN GAP and optimizer', 'crop_fractions': [0.15, 0, 0.85, 0.9], 'fixed_combination_choices': {'global_crop_weights': [0, 0.5, 1], 'face_weights': [0, 1], 'calibration': ['scalar_temperature', 'regularized_class_bias'], 'class_bias_l2': 0.01}, 'notes': ['Each OOF image is predicted only by a model excluding its driver.', 'Fixed epoch configuration was informed by prior experiments on the original fold0; results remain development estimates.', 'Calibrator crossfit is not a full nested re-training of base models.', 'Test predictions are transformed only after validation-based choices; no pseudo labels or test fitting.'], 'new_competition_submissions_authorized': 0, 'family': 'full', 'face_oof': True}
DRIVER_CROP = None
# %% Configuration and data
import copy
import gc
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from scipy.optimize import minimize_scalar

ARCH = 'vgg16_bn_gap_driver_cv'
SEED = 42
MAX_EPOCHS = 9
PATIENCE = 3
BATCH_SIZE = 64
WORKERS = 4
LABEL_SMOOTHING = 0.03
EMA_DECAY = 0.995
INPUT = INPUT_ROOT
OUT = WORK_ROOT/'full';OUT.mkdir(parents=True,exist_ok=True)
CLASSES = [f'c{i}' for i in range(10)]
STARTED = time.perf_counter()
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required.')
DEVICE = torch.device('cuda:0')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def emit(event, **fields):
    print(json.dumps({'event':event, 'arch':ARCH, **fields}), flush=True)

def find_input(name):
    paths=set()
    for prefix in ('','*/','*/*/'):
        paths.update(INPUT.glob(prefix+name))
    if len(paths)!=1:
        raise FileNotFoundError(f'Expected one {name}: {sorted(paths)}')
    return next(iter(paths))

metadata_path=find_input('driver_imgs_list.csv')
assert hashlib.sha256(metadata_path.read_bytes()).hexdigest()==CV_CONFIG['metadata_sha256']
metadata=pd.read_csv(metadata_path)
metadata['label']=metadata['classname'].str[1:].astype(int)
assert set(metadata['label'])==set(range(10)) and not metadata['img'].duplicated().any()
DATA_ROOT=metadata_path.parent
TRAIN_ROOT=DATA_ROOT/'imgs/train'
TEST_ROOT=DATA_ROOT/'imgs/test'
assert TRAIN_ROOT.is_dir() and TEST_ROOT.is_dir()
normalize=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomAffine(degrees=8,translate=(.05,.05),scale=(.9,1.1),fill=(123,116,103)),
    transforms.ColorJitter(brightness=.2,contrast=.2,saturation=.1),
    transforms.ToTensor(),normalize])
eval_transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),normalize])

class DriverDataset(Dataset):
    def __init__(self,frame,training=False,test=False):
        self.frame=frame.reset_index(drop=True)
        self.transform=train_transform if training else eval_transform
        self.test=test
    def __len__(self):
        return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        path=TEST_ROOT/row['img'] if self.test else TRAIN_ROOT/row['classname']/row['img']
        with Image.open(path) as image:
            image=image.convert('RGB')
            if DRIVER_CROP is not None:
                w,h=image.size
                a,b,c,d=DRIVER_CROP
                image=image.crop((int(a*w),int(b*h),int(c*w),int(d*h)))
            image=self.transform(image)
        return image,index if self.test else int(row['label'])

def seed_worker(worker_id):
    seed=torch.initial_seed()%(2**32)
    random.seed(seed); np.random.seed(seed)

def loader(frame,training=False,test=False):
    return DataLoader(DriverDataset(frame,training,test),batch_size=BATCH_SIZE,shuffle=training,
        num_workers=WORKERS,pin_memory=True,persistent_workers=False,worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED))

def prediction_frame(frame,prob):
    result=frame[['subject','classname','img','label']].copy()
    result[CLASSES]=prob
    return result

# %% Model, optimization and inference
class VGG16BNGAP(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.features=features
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.classifier=nn.Sequential(nn.Identity(),nn.Identity(),nn.Identity(),nn.Identity(),
                                      nn.Identity(),nn.Dropout(.3),nn.Linear(512,10))
    def forward(self,x):
        return self.classifier(self.pool(self.features(x)).flatten(1))

def make_model():
    seed_all(SEED)
    pretrained=models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
    core=VGG16BNGAP(pretrained.features).to(DEVICE)
    del pretrained
    gc.collect()
    wrapped=nn.DataParallel(core) if torch.cuda.device_count()>1 else core
    return core,wrapped,24

def make_optimizer(core,freeze_until):
    return torch.optim.AdamW([
        {'params':list(core.features[freeze_until:].parameters()),'lr':0},
        {'params':list(core.classifier[:6].parameters()),'lr':3e-5},
        {'params':list(core.classifier[6].parameters()),'lr':3e-4}],weight_decay=1e-4)

def configure_epoch(core,model,optimizer,freeze_until,epoch):
    model.train()
    core.features.requires_grad_(False)
    if epoch==1:
        core.features.eval()
        rates=[0,0,1e-3]
    else:
        core.features[freeze_until:].requires_grad_(True)
        core.features[:freeze_until].eval()
        phase=(epoch-2)/max(1,MAX_EPOCHS-2)
        factor=.1+.9*.5*(1+math.cos(math.pi*phase))
        rates=[3e-5*factor,0,3e-4*factor]
    for group,rate in zip(optimizer.param_groups,rates):group['lr']=rate
    return rates

@torch.no_grad()
def update_ema(ema,core):
    for average,current in zip(ema.parameters(),core.parameters()):
        average.lerp_(current.detach(),1-EMA_DECAY)
    for average,current in zip(ema.buffers(),core.buffers()):
        average.copy_(current.detach())

def train_epoch(core,model,optimizer,scaler,train_loader,ema,stage,epoch):
    loss_fn=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    total=correct=0
    loss_sum=0.
    for batch,(images,targets) in enumerate(train_loader):
        images=images.to(DEVICE,non_blocking=True); targets=targets.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):logits=model(images)
        loss=loss_fn(logits.float(),targets)
        if not torch.isfinite(loss):raise RuntimeError('Non-finite training loss.')
        scaler.scale(loss).backward()
        if epoch==2 and batch==0:
            convs=[m for m in core.features if isinstance(m,nn.Conv2d)]
            assert convs[-1].weight.grad is not None and convs[0].weight.grad is None
            emit('gradient_check_passed',stage=stage)
        scaler.step(optimizer);scaler.update()
        if ema is not None:update_ema(ema,core)
        total+=len(targets); correct+=(logits.argmax(1)==targets).sum().item()
        loss_sum+=loss.item()*len(targets)
        if (batch+1)%100==0:
            emit('training_progress',stage=stage,epoch=epoch,batch=batch+1,batches=len(train_loader),train_accuracy=correct/total)
    return loss_sum/total,correct/total

def predict(model,data_loader,views=1,test=False):
    model.eval()
    results=[]; targets=[]
    offsets=[0] if views==1 else [0,-4,4]
    with torch.inference_mode():
        for batch,(images,label) in enumerate(data_loader):
            images=images.to(DEVICE,non_blocking=True)
            if views>1: padded=nn.functional.pad(images,(4,4,0,0),value=0)
            summed=None
            for offset in offsets:
                current=images if views==1 else padded[:,:,:,4+offset:4+offset+224].contiguous()
                with torch.autocast('cuda',dtype=torch.float16):logits=model(current)
                prob=logits.double().softmax(1)
                summed=prob if summed is None else summed+prob
            results.append((summed/len(offsets)).cpu().numpy())
            targets.append(label.numpy())
            if test and (batch+1)%200==0:
                emit('test_progress',completed_images=sum(len(x) for x in results),total_images=len(data_loader.dataset),views=views)
    results=np.concatenate(results);targets=np.concatenate(targets)
    assert np.isfinite(results).all()
    np.testing.assert_allclose(results.sum(axis=1),1,atol=1e-8)
    return targets,results

def evaluate(model,data_loader,views=1):
    targets,prob=predict(model,data_loader,views)
    np.testing.assert_array_equal(targets,val['label'].to_numpy())
    return float(log_loss(targets,prob,labels=list(range(10)))),float(accuracy_score(targets,prob.argmax(1))),prob

def save_model(core,path,report):
    state={k:v.detach().cpu() for k,v in core.state_dict().items()}
    torch.save({'model_state_dict':state,'arch':ARCH,'metadata':report},path)

def temperature_transform(prob,temperature):
    z=np.log(np.clip(prob,1e-15,1))/temperature
    z-=z.max(axis=1,keepdims=True)
    z=np.exp(z)
    return z/z.sum(axis=1,keepdims=True)

def calibrate(prob):
    labels=val['label'].to_numpy()
    def fit(mask):
        result=minimize_scalar(lambda z:log_loss(labels[mask],temperature_transform(prob[mask],np.exp(z)),labels=list(range(10))),
            bounds=(np.log(.5),np.log(3)),method='bounded')
        return float(np.exp(result.x))
    drivers=sorted(val['subject'].unique())
    crossfit=np.zeros_like(prob);folds=[]
    for held_out in (drivers[::2],drivers[1::2]):
        test_mask=val['subject'].isin(held_out).to_numpy();train_mask=~test_mask
        temperature=fit(train_mask)
        crossfit[test_mask]=temperature_transform(prob[test_mask],temperature)
        folds.append({'evaluation_drivers':held_out,'temperature':temperature,
            'raw_log_loss':float(log_loss(labels[test_mask],prob[test_mask],labels=list(range(10)))),
            'calibrated_log_loss':float(log_loss(labels[test_mask],crossfit[test_mask],labels=list(range(10))))})
    raw_score=float(log_loss(labels,prob,labels=list(range(10))))
    crossfit_score=float(log_loss(labels,crossfit,labels=list(range(10))))
    enabled=crossfit_score<raw_score-.003 and all(f['calibrated_log_loss']<=f['raw_log_loss']+.02 for f in folds)
    temperature=fit(np.ones(len(val),dtype=bool)) if enabled else 1.
    if not enabled:crossfit=prob.copy();crossfit_score=raw_score
    return temperature,crossfit,{'enabled':enabled,'temperature':temperature,'raw_log_loss':raw_score,
        'crossfit_log_loss':crossfit_score,'folds':folds}


"""Fixed-epoch, driver-disjoint out-of-fold training and test averaging.

There is no epoch/weight selection using the current fold's labels. The fixed
5-epoch EMA configuration was informed by earlier development trials, including
the original six-driver fold, so these are still development estimates.
"""
FOLD_IDS=metadata.subject.map(CV_CONFIG['driver_folds']).to_numpy()
assert set(FOLD_IDS)==set(range(5)) and len(metadata)==22424
assert metadata.subject.nunique()==26
EPOCHS=5
CLASSES=[f'c{i}' for i in range(10)]
oof=np.full((len(metadata),10),np.nan)
seen=np.zeros(len(metadata),dtype=np.int8)
face_oof=np.full(len(metadata),np.nan)
sample=pd.read_csv(find_input('sample_submission.csv'))
assert sample.columns.tolist()==['img']+CLASSES and len(sample)==79726
test_sum=np.zeros((len(sample),10),dtype=np.float64)
manifest=metadata.copy();manifest['fold']=FOLD_IDS
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
(OUT/'cv_protocol.json').write_text(json.dumps(CV_CONFIG,indent=2))

class FaceData(Dataset):
    def __init__(self,frame,training=False):
        self.frame=frame.reset_index(drop=True);self.training=training
        self.transform=transforms.Compose([transforms.Resize((224,224)),
            transforms.RandomAffine(degrees=5,translate=(.04,.04),scale=(.95,1.05),fill=(123,116,103)),
            transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.1),
            transforms.ToTensor(),normalize]) if training else eval_transform
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with Image.open(TRAIN_ROOT/row.classname/row.img) as image:
            image=image.convert('RGB');w,h=image.size
            image=image.crop((int(.2*w),0,int(.7*w),int(.65*h)))
            image=self.transform(image)
        return image,int(row.label==9)

def face_loader(frame,training=False):
    return DataLoader(FaceData(frame,training),batch_size=32 if training else 64,shuffle=training,
        num_workers=4,pin_memory=True,worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED))

class FaceVGG(nn.Module):
    def __init__(self,features):
        super().__init__();self.features=features
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(nn.Dropout(.3),nn.Linear(512,2))
    def forward(self,x):return self.classifier(self.pool(self.features(x)).flatten(1))

def fit_face(train_frame,val_frame,fold_dir,fold):
    seed_all(SEED)
    pre=models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    core=FaceVGG(pre.features).to(DEVICE);del pre;gc.collect()
    model=nn.DataParallel(core) if torch.cuda.device_count()>1 else core
    opt=torch.optim.AdamW([{'params':list(core.features[17:].parameters()),'lr':0},
                          {'params':core.classifier.parameters(),'lr':1e-3}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda')
    binary=train_frame[train_frame.label.isin([0,9])].reset_index(drop=True)
    assert not set(binary.subject)&set(val_frame.subject)
    dl=face_loader(binary,True)
    for epoch in [1,2]:
        model.train();core.features.requires_grad_(False)
        if epoch==2:core.features[17:].requires_grad_(True)
        rates=[0,1e-3] if epoch==1 else [3e-5,2e-4]
        for group,lr in zip(opt.param_groups,rates):group['lr']=lr
        n=0;total=0.
        for x,y in dl:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):z=model(x)
            loss=nn.functional.cross_entropy(z.float(),y,label_smoothing=.02)
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.step(opt);scaler.update()
            n+=len(y);total+=float(loss)*len(y)
        emit('face_epoch',fold=fold,epoch=epoch,images=n,loss=total/n)
    model.eval();qs=[]
    with torch.inference_mode():
        for x,y in face_loader(val_frame):
            with torch.autocast('cuda',dtype=torch.float16):z=model(x.to(DEVICE,non_blocking=True))
            qs.append(z.double().softmax(1)[:,1].cpu().numpy())
    q=np.concatenate(qs)
    assert len(q)==len(val_frame) and np.isfinite(q).all()
    frame=val_frame[['subject','classname','img','label']].copy();frame['face_c9']=q
    frame.to_csv(fold_dir/'face_oof_predictions.csv',index=False)
    torch.save({'model_state_dict':{k:v.detach().cpu() for k,v in core.state_dict().items()},
                'fold':fold,'epochs':2,'train_drivers':sorted(binary.subject.unique().tolist())},fold_dir/'face_model.pt')
    del core,model,opt,scaler,dl;gc.collect();torch.cuda.empty_cache()
    return q

reports=[]
emit('cv_started',family=CV_CONFIG['family'],folds=5,epochs=EPOCHS,
     training_schedule_epochs=MAX_EPOCHS,crop=DRIVER_CROP,face_oof=CV_CONFIG['face_oof'])
for fold in range(5):
    start=time.perf_counter();SEED=42+1000*fold
    fit_indices=np.flatnonzero(FOLD_IDS!=fold);val_indices=np.flatnonzero(FOLD_IDS==fold)
    train=metadata.iloc[fit_indices].reset_index(drop=True)
    val=metadata.iloc[val_indices].reset_index(drop=True)
    assert not set(train.subject)&set(val.subject)
    dest=OUT/f'fold_{fold}';dest.mkdir(exist_ok=True)
    train.to_csv(dest/'train_manifest.csv',index=False);val.to_csv(dest/'validation_manifest.csv',index=False)
    core,model,freeze_until=make_model();opt=make_optimizer(core,freeze_until)
    scaler=torch.amp.GradScaler('cuda');dl=loader(train,training=True);ema=None;history=[]
    emit('fold_started',family=CV_CONFIG['family'],fold=fold,fit_images=len(train),val_images=len(val),
         fit_drivers=sorted(train.subject.unique().tolist()),val_drivers=sorted(val.subject.unique().tolist()))
    for epoch in range(1,EPOCHS+1):
        epoch_start=time.perf_counter()
        configure_epoch(core,model,opt,freeze_until,epoch)
        loss,accuracy=train_epoch(core,model,opt,scaler,dl,ema,f'fold_{fold}',epoch)
        if epoch==1:ema=copy.deepcopy(core).requires_grad_(False).eval()
        row={'fold':fold,'epoch':epoch,'train_loss':loss,'train_accuracy':accuracy,'seconds':time.perf_counter()-epoch_start}
        history.append(row);pd.DataFrame(history).to_csv(dest/'training_history.csv',index=False)
        emit('cv_epoch',family=CV_CONFIG['family'],**row)
    core.load_state_dict(ema.state_dict())
    emit('bn_refresh_started',family=CV_CONFIG['family'],fold=fold,data='fit drivers only')
    from torch.optim.swa_utils import update_bn
    update_bn(loader(train),core,device=DEVICE)
    save_model(core,dest/'global_model.pt',{'fold':fold,'config':CV_CONFIG,'epochs':EPOCHS,'variant':'ema',
                                         'train_drivers':sorted(train.subject.unique().tolist())})
    core.requires_grad_(False);core.zero_grad(set_to_none=True)
    del opt,scaler,dl,ema;gc.collect();torch.cuda.empty_cache()
    labels,prob=predict(model,loader(val))
    np.testing.assert_array_equal(labels,val.label.to_numpy())
    assert not seen[val_indices].any()
    oof[val_indices]=prob;seen[val_indices]+=1
    frame=prediction_frame(val,prob);frame['fold']=fold;frame.to_csv(dest/'oof_predictions.csv',index=False)
    row={'fold':fold,'fit_drivers':sorted(train.subject.unique().tolist()),'validation_drivers':sorted(val.subject.unique().tolist()),
         'validation_log_loss':float(log_loss(val.label,prob,labels=list(range(10)))),
         'validation_accuracy':float(accuracy_score(val.label,prob.argmax(1))),
         'epochs_fixed_before_training':EPOCHS,'validation_used_for_checkpoint_selection':False}
    emit('fold_validation',family=CV_CONFIG['family'],**row)
    indices,test_prob=predict(model,loader(sample,test=True),test=True)
    np.testing.assert_array_equal(indices,np.arange(len(sample)))
    test_sum+=test_prob
    test_frame=sample[['img']].copy();test_frame[CLASSES]=test_prob
    test_frame.to_csv(dest/'test_raw.csv',index=False,float_format='%.12g')
    del core,model,test_prob;gc.collect();torch.cuda.empty_cache()
    if CV_CONFIG['face_oof']:face_oof[val_indices]=fit_face(train,val,dest,fold)
    row['seconds']=time.perf_counter()-start;reports.append(row)
    (OUT/'fold_reports.json').write_text(json.dumps(reports,indent=2))
    emit('fold_complete',family=CV_CONFIG['family'],**row)
assert (seen==1).all() and np.isfinite(oof).all()
np.testing.assert_allclose(oof.sum(1),1,atol=1e-8)
result=metadata[['subject','classname','img','label']].copy();result['fold']=FOLD_IDS;result[CLASSES]=oof
result.to_csv(OUT/'oof_predictions.csv',index=False)
test_frame=sample[['img']].copy();test_frame[CLASSES]=test_sum/5
test_frame.to_csv(OUT/'test_mean_raw.csv',index=False,float_format='%.12g')
if CV_CONFIG['face_oof']:
    assert np.isfinite(face_oof).all() and (face_oof>=0).all() and (face_oof<=1).all()
    face_frame=metadata[['subject','classname','img','label']].copy();face_frame['fold']=FOLD_IDS;face_frame['face_c9']=face_oof
    face_frame.to_csv(OUT/'face_oof_predictions.csv',index=False)
report={'family':CV_CONFIG['family'],'oof_images':len(metadata),'drivers':int(metadata.subject.nunique()),
        'oof_log_loss':float(log_loss(metadata.label,oof,labels=list(range(10)))),
        'oof_accuracy':float(accuracy_score(metadata.label,oof.argmax(1))),
        'test_images':len(sample),'test_models_averaged':5,'folds':reports,
        'every_oof_driver_excluded_from_its_training_model':True,
        'test_labels_used':False,'automatically_submitted':False,'config':CV_CONFIG,
        'elapsed_seconds':time.perf_counter()-STARTED}
(OUT/'cv_report.json').write_text(json.dumps(report,indent=2))
emit('cv_complete',**report)


## 2. 운전자 영역 크롭 VGG 5-fold

In [ ]:
from pathlib import Path
import os
INPUT_ROOT=Path(os.environ.get('STATEFARM_INPUT_ROOT','/kaggle/input'))
_default_output='/kaggle/working/vgg_final' if Path('/kaggle/working').is_dir() else './vgg_final_outputs'
WORK_ROOT=Path(os.environ.get('STATEFARM_OUTPUT_ROOT',_default_output))
WORK_ROOT.mkdir(parents=True,exist_ok=True)
CV_CONFIG = {'driver_folds': {'p002': 0, 'p026': 0, 'p035': 0, 'p041': 0, 'p050': 0, 'p075': 0, 'p081': 1, 'p051': 1, 'p015': 1, 'p024': 1, 'p012': 1, 'p014': 2, 'p056': 2, 'p052': 2, 'p016': 2, 'p061': 2, 'p064': 3, 'p047': 3, 'p072': 3, 'p022': 3, 'p045': 3, 'p066': 4, 'p039': 4, 'p049': 4, 'p021': 4, 'p042': 4}, 'metadata_sha256': '891ca8e2bfd7a3522ac2098a1bc93825c581d07104d211c329de0b16c83ca5ee', 'fixed_epochs': 5, 'schedule_epochs': 9, 'fixed_variant': 'EMA .995; BN refreshed on fit drivers only', 'face_fixed_epochs': 2, 'family_comparison': 'full image vs fixed driver crop; same VGG16-BN GAP and optimizer', 'crop_fractions': [0.15, 0, 0.85, 0.9], 'fixed_combination_choices': {'global_crop_weights': [0, 0.5, 1], 'face_weights': [0, 1], 'calibration': ['scalar_temperature', 'regularized_class_bias'], 'class_bias_l2': 0.01}, 'notes': ['Each OOF image is predicted only by a model excluding its driver.', 'Fixed epoch configuration was informed by prior experiments on the original fold0; results remain development estimates.', 'Calibrator crossfit is not a full nested re-training of base models.', 'Test predictions are transformed only after validation-based choices; no pseudo labels or test fitting.'], 'new_competition_submissions_authorized': 0, 'family': 'crop', 'face_oof': False}
DRIVER_CROP = [0.15, 0, 0.85, 0.9]
# %% Configuration and data
import copy
import gc
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupShuffleSplit
from scipy.optimize import minimize_scalar

ARCH = 'vgg16_bn_gap_driver_cv'
SEED = 42
MAX_EPOCHS = 9
PATIENCE = 3
BATCH_SIZE = 64
WORKERS = 4
LABEL_SMOOTHING = 0.03
EMA_DECAY = 0.995
INPUT = INPUT_ROOT
OUT = WORK_ROOT/'crop';OUT.mkdir(parents=True,exist_ok=True)
CLASSES = [f'c{i}' for i in range(10)]
STARTED = time.perf_counter()
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required.')
DEVICE = torch.device('cuda:0')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def emit(event, **fields):
    print(json.dumps({'event':event, 'arch':ARCH, **fields}), flush=True)

def find_input(name):
    paths=set()
    for prefix in ('','*/','*/*/'):
        paths.update(INPUT.glob(prefix+name))
    if len(paths)!=1:
        raise FileNotFoundError(f'Expected one {name}: {sorted(paths)}')
    return next(iter(paths))

metadata_path=find_input('driver_imgs_list.csv')
assert hashlib.sha256(metadata_path.read_bytes()).hexdigest()==CV_CONFIG['metadata_sha256']
metadata=pd.read_csv(metadata_path)
metadata['label']=metadata['classname'].str[1:].astype(int)
assert set(metadata['label'])==set(range(10)) and not metadata['img'].duplicated().any()
DATA_ROOT=metadata_path.parent
TRAIN_ROOT=DATA_ROOT/'imgs/train'
TEST_ROOT=DATA_ROOT/'imgs/test'
assert TRAIN_ROOT.is_dir() and TEST_ROOT.is_dir()
normalize=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomAffine(degrees=8,translate=(.05,.05),scale=(.9,1.1),fill=(123,116,103)),
    transforms.ColorJitter(brightness=.2,contrast=.2,saturation=.1),
    transforms.ToTensor(),normalize])
eval_transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),normalize])

class DriverDataset(Dataset):
    def __init__(self,frame,training=False,test=False):
        self.frame=frame.reset_index(drop=True)
        self.transform=train_transform if training else eval_transform
        self.test=test
    def __len__(self):
        return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        path=TEST_ROOT/row['img'] if self.test else TRAIN_ROOT/row['classname']/row['img']
        with Image.open(path) as image:
            image=image.convert('RGB')
            if DRIVER_CROP is not None:
                w,h=image.size
                a,b,c,d=DRIVER_CROP
                image=image.crop((int(a*w),int(b*h),int(c*w),int(d*h)))
            image=self.transform(image)
        return image,index if self.test else int(row['label'])

def seed_worker(worker_id):
    seed=torch.initial_seed()%(2**32)
    random.seed(seed); np.random.seed(seed)

def loader(frame,training=False,test=False):
    return DataLoader(DriverDataset(frame,training,test),batch_size=BATCH_SIZE,shuffle=training,
        num_workers=WORKERS,pin_memory=True,persistent_workers=False,worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED))

def prediction_frame(frame,prob):
    result=frame[['subject','classname','img','label']].copy()
    result[CLASSES]=prob
    return result

# %% Model, optimization and inference
class VGG16BNGAP(nn.Module):
    def __init__(self,features):
        super().__init__()
        self.features=features
        self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.classifier=nn.Sequential(nn.Identity(),nn.Identity(),nn.Identity(),nn.Identity(),
                                      nn.Identity(),nn.Dropout(.3),nn.Linear(512,10))
    def forward(self,x):
        return self.classifier(self.pool(self.features(x)).flatten(1))

def make_model():
    seed_all(SEED)
    pretrained=models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1)
    core=VGG16BNGAP(pretrained.features).to(DEVICE)
    del pretrained
    gc.collect()
    wrapped=nn.DataParallel(core) if torch.cuda.device_count()>1 else core
    return core,wrapped,24

def make_optimizer(core,freeze_until):
    return torch.optim.AdamW([
        {'params':list(core.features[freeze_until:].parameters()),'lr':0},
        {'params':list(core.classifier[:6].parameters()),'lr':3e-5},
        {'params':list(core.classifier[6].parameters()),'lr':3e-4}],weight_decay=1e-4)

def configure_epoch(core,model,optimizer,freeze_until,epoch):
    model.train()
    core.features.requires_grad_(False)
    if epoch==1:
        core.features.eval()
        rates=[0,0,1e-3]
    else:
        core.features[freeze_until:].requires_grad_(True)
        core.features[:freeze_until].eval()
        phase=(epoch-2)/max(1,MAX_EPOCHS-2)
        factor=.1+.9*.5*(1+math.cos(math.pi*phase))
        rates=[3e-5*factor,0,3e-4*factor]
    for group,rate in zip(optimizer.param_groups,rates):group['lr']=rate
    return rates

@torch.no_grad()
def update_ema(ema,core):
    for average,current in zip(ema.parameters(),core.parameters()):
        average.lerp_(current.detach(),1-EMA_DECAY)
    for average,current in zip(ema.buffers(),core.buffers()):
        average.copy_(current.detach())

def train_epoch(core,model,optimizer,scaler,train_loader,ema,stage,epoch):
    loss_fn=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    total=correct=0
    loss_sum=0.
    for batch,(images,targets) in enumerate(train_loader):
        images=images.to(DEVICE,non_blocking=True); targets=targets.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):logits=model(images)
        loss=loss_fn(logits.float(),targets)
        if not torch.isfinite(loss):raise RuntimeError('Non-finite training loss.')
        scaler.scale(loss).backward()
        if epoch==2 and batch==0:
            convs=[m for m in core.features if isinstance(m,nn.Conv2d)]
            assert convs[-1].weight.grad is not None and convs[0].weight.grad is None
            emit('gradient_check_passed',stage=stage)
        scaler.step(optimizer);scaler.update()
        if ema is not None:update_ema(ema,core)
        total+=len(targets); correct+=(logits.argmax(1)==targets).sum().item()
        loss_sum+=loss.item()*len(targets)
        if (batch+1)%100==0:
            emit('training_progress',stage=stage,epoch=epoch,batch=batch+1,batches=len(train_loader),train_accuracy=correct/total)
    return loss_sum/total,correct/total

def predict(model,data_loader,views=1,test=False):
    model.eval()
    results=[]; targets=[]
    offsets=[0] if views==1 else [0,-4,4]
    with torch.inference_mode():
        for batch,(images,label) in enumerate(data_loader):
            images=images.to(DEVICE,non_blocking=True)
            if views>1: padded=nn.functional.pad(images,(4,4,0,0),value=0)
            summed=None
            for offset in offsets:
                current=images if views==1 else padded[:,:,:,4+offset:4+offset+224].contiguous()
                with torch.autocast('cuda',dtype=torch.float16):logits=model(current)
                prob=logits.double().softmax(1)
                summed=prob if summed is None else summed+prob
            results.append((summed/len(offsets)).cpu().numpy())
            targets.append(label.numpy())
            if test and (batch+1)%200==0:
                emit('test_progress',completed_images=sum(len(x) for x in results),total_images=len(data_loader.dataset),views=views)
    results=np.concatenate(results);targets=np.concatenate(targets)
    assert np.isfinite(results).all()
    np.testing.assert_allclose(results.sum(axis=1),1,atol=1e-8)
    return targets,results

def evaluate(model,data_loader,views=1):
    targets,prob=predict(model,data_loader,views)
    np.testing.assert_array_equal(targets,val['label'].to_numpy())
    return float(log_loss(targets,prob,labels=list(range(10)))),float(accuracy_score(targets,prob.argmax(1))),prob

def save_model(core,path,report):
    state={k:v.detach().cpu() for k,v in core.state_dict().items()}
    torch.save({'model_state_dict':state,'arch':ARCH,'metadata':report},path)

def temperature_transform(prob,temperature):
    z=np.log(np.clip(prob,1e-15,1))/temperature
    z-=z.max(axis=1,keepdims=True)
    z=np.exp(z)
    return z/z.sum(axis=1,keepdims=True)

def calibrate(prob):
    labels=val['label'].to_numpy()
    def fit(mask):
        result=minimize_scalar(lambda z:log_loss(labels[mask],temperature_transform(prob[mask],np.exp(z)),labels=list(range(10))),
            bounds=(np.log(.5),np.log(3)),method='bounded')
        return float(np.exp(result.x))
    drivers=sorted(val['subject'].unique())
    crossfit=np.zeros_like(prob);folds=[]
    for held_out in (drivers[::2],drivers[1::2]):
        test_mask=val['subject'].isin(held_out).to_numpy();train_mask=~test_mask
        temperature=fit(train_mask)
        crossfit[test_mask]=temperature_transform(prob[test_mask],temperature)
        folds.append({'evaluation_drivers':held_out,'temperature':temperature,
            'raw_log_loss':float(log_loss(labels[test_mask],prob[test_mask],labels=list(range(10)))),
            'calibrated_log_loss':float(log_loss(labels[test_mask],crossfit[test_mask],labels=list(range(10))))})
    raw_score=float(log_loss(labels,prob,labels=list(range(10))))
    crossfit_score=float(log_loss(labels,crossfit,labels=list(range(10))))
    enabled=crossfit_score<raw_score-.003 and all(f['calibrated_log_loss']<=f['raw_log_loss']+.02 for f in folds)
    temperature=fit(np.ones(len(val),dtype=bool)) if enabled else 1.
    if not enabled:crossfit=prob.copy();crossfit_score=raw_score
    return temperature,crossfit,{'enabled':enabled,'temperature':temperature,'raw_log_loss':raw_score,
        'crossfit_log_loss':crossfit_score,'folds':folds}


"""Fixed-epoch, driver-disjoint out-of-fold training and test averaging.

There is no epoch/weight selection using the current fold's labels. The fixed
5-epoch EMA configuration was informed by earlier development trials, including
the original six-driver fold, so these are still development estimates.
"""
FOLD_IDS=metadata.subject.map(CV_CONFIG['driver_folds']).to_numpy()
assert set(FOLD_IDS)==set(range(5)) and len(metadata)==22424
assert metadata.subject.nunique()==26
EPOCHS=5
CLASSES=[f'c{i}' for i in range(10)]
oof=np.full((len(metadata),10),np.nan)
seen=np.zeros(len(metadata),dtype=np.int8)
face_oof=np.full(len(metadata),np.nan)
sample=pd.read_csv(find_input('sample_submission.csv'))
assert sample.columns.tolist()==['img']+CLASSES and len(sample)==79726
test_sum=np.zeros((len(sample),10),dtype=np.float64)
manifest=metadata.copy();manifest['fold']=FOLD_IDS
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
(OUT/'cv_protocol.json').write_text(json.dumps(CV_CONFIG,indent=2))

class FaceData(Dataset):
    def __init__(self,frame,training=False):
        self.frame=frame.reset_index(drop=True);self.training=training
        self.transform=transforms.Compose([transforms.Resize((224,224)),
            transforms.RandomAffine(degrees=5,translate=(.04,.04),scale=(.95,1.05),fill=(123,116,103)),
            transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.1),
            transforms.ToTensor(),normalize]) if training else eval_transform
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        with Image.open(TRAIN_ROOT/row.classname/row.img) as image:
            image=image.convert('RGB');w,h=image.size
            image=image.crop((int(.2*w),0,int(.7*w),int(.65*h)))
            image=self.transform(image)
        return image,int(row.label==9)

def face_loader(frame,training=False):
    return DataLoader(FaceData(frame,training),batch_size=32 if training else 64,shuffle=training,
        num_workers=4,pin_memory=True,worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED))

class FaceVGG(nn.Module):
    def __init__(self,features):
        super().__init__();self.features=features
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(nn.Dropout(.3),nn.Linear(512,2))
    def forward(self,x):return self.classifier(self.pool(self.features(x)).flatten(1))

def fit_face(train_frame,val_frame,fold_dir,fold):
    seed_all(SEED)
    pre=models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    core=FaceVGG(pre.features).to(DEVICE);del pre;gc.collect()
    model=nn.DataParallel(core) if torch.cuda.device_count()>1 else core
    opt=torch.optim.AdamW([{'params':list(core.features[17:].parameters()),'lr':0},
                          {'params':core.classifier.parameters(),'lr':1e-3}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda')
    binary=train_frame[train_frame.label.isin([0,9])].reset_index(drop=True)
    assert not set(binary.subject)&set(val_frame.subject)
    dl=face_loader(binary,True)
    for epoch in [1,2]:
        model.train();core.features.requires_grad_(False)
        if epoch==2:core.features[17:].requires_grad_(True)
        rates=[0,1e-3] if epoch==1 else [3e-5,2e-4]
        for group,lr in zip(opt.param_groups,rates):group['lr']=lr
        n=0;total=0.
        for x,y in dl:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):z=model(x)
            loss=nn.functional.cross_entropy(z.float(),y,label_smoothing=.02)
            assert torch.isfinite(loss)
            scaler.scale(loss).backward();scaler.step(opt);scaler.update()
            n+=len(y);total+=float(loss)*len(y)
        emit('face_epoch',fold=fold,epoch=epoch,images=n,loss=total/n)
    model.eval();qs=[]
    with torch.inference_mode():
        for x,y in face_loader(val_frame):
            with torch.autocast('cuda',dtype=torch.float16):z=model(x.to(DEVICE,non_blocking=True))
            qs.append(z.double().softmax(1)[:,1].cpu().numpy())
    q=np.concatenate(qs)
    assert len(q)==len(val_frame) and np.isfinite(q).all()
    frame=val_frame[['subject','classname','img','label']].copy();frame['face_c9']=q
    frame.to_csv(fold_dir/'face_oof_predictions.csv',index=False)
    torch.save({'model_state_dict':{k:v.detach().cpu() for k,v in core.state_dict().items()},
                'fold':fold,'epochs':2,'train_drivers':sorted(binary.subject.unique().tolist())},fold_dir/'face_model.pt')
    del core,model,opt,scaler,dl;gc.collect();torch.cuda.empty_cache()
    return q

reports=[]
emit('cv_started',family=CV_CONFIG['family'],folds=5,epochs=EPOCHS,
     training_schedule_epochs=MAX_EPOCHS,crop=DRIVER_CROP,face_oof=CV_CONFIG['face_oof'])
for fold in range(5):
    start=time.perf_counter();SEED=42+1000*fold
    fit_indices=np.flatnonzero(FOLD_IDS!=fold);val_indices=np.flatnonzero(FOLD_IDS==fold)
    train=metadata.iloc[fit_indices].reset_index(drop=True)
    val=metadata.iloc[val_indices].reset_index(drop=True)
    assert not set(train.subject)&set(val.subject)
    dest=OUT/f'fold_{fold}';dest.mkdir(exist_ok=True)
    train.to_csv(dest/'train_manifest.csv',index=False);val.to_csv(dest/'validation_manifest.csv',index=False)
    core,model,freeze_until=make_model();opt=make_optimizer(core,freeze_until)
    scaler=torch.amp.GradScaler('cuda');dl=loader(train,training=True);ema=None;history=[]
    emit('fold_started',family=CV_CONFIG['family'],fold=fold,fit_images=len(train),val_images=len(val),
         fit_drivers=sorted(train.subject.unique().tolist()),val_drivers=sorted(val.subject.unique().tolist()))
    for epoch in range(1,EPOCHS+1):
        epoch_start=time.perf_counter()
        configure_epoch(core,model,opt,freeze_until,epoch)
        loss,accuracy=train_epoch(core,model,opt,scaler,dl,ema,f'fold_{fold}',epoch)
        if epoch==1:ema=copy.deepcopy(core).requires_grad_(False).eval()
        row={'fold':fold,'epoch':epoch,'train_loss':loss,'train_accuracy':accuracy,'seconds':time.perf_counter()-epoch_start}
        history.append(row);pd.DataFrame(history).to_csv(dest/'training_history.csv',index=False)
        emit('cv_epoch',family=CV_CONFIG['family'],**row)
    core.load_state_dict(ema.state_dict())
    emit('bn_refresh_started',family=CV_CONFIG['family'],fold=fold,data='fit drivers only')
    from torch.optim.swa_utils import update_bn
    update_bn(loader(train),core,device=DEVICE)
    save_model(core,dest/'global_model.pt',{'fold':fold,'config':CV_CONFIG,'epochs':EPOCHS,'variant':'ema',
                                         'train_drivers':sorted(train.subject.unique().tolist())})
    core.requires_grad_(False);core.zero_grad(set_to_none=True)
    del opt,scaler,dl,ema;gc.collect();torch.cuda.empty_cache()
    labels,prob=predict(model,loader(val))
    np.testing.assert_array_equal(labels,val.label.to_numpy())
    assert not seen[val_indices].any()
    oof[val_indices]=prob;seen[val_indices]+=1
    frame=prediction_frame(val,prob);frame['fold']=fold;frame.to_csv(dest/'oof_predictions.csv',index=False)
    row={'fold':fold,'fit_drivers':sorted(train.subject.unique().tolist()),'validation_drivers':sorted(val.subject.unique().tolist()),
         'validation_log_loss':float(log_loss(val.label,prob,labels=list(range(10)))),
         'validation_accuracy':float(accuracy_score(val.label,prob.argmax(1))),
         'epochs_fixed_before_training':EPOCHS,'validation_used_for_checkpoint_selection':False}
    emit('fold_validation',family=CV_CONFIG['family'],**row)
    indices,test_prob=predict(model,loader(sample,test=True),test=True)
    np.testing.assert_array_equal(indices,np.arange(len(sample)))
    test_sum+=test_prob
    test_frame=sample[['img']].copy();test_frame[CLASSES]=test_prob
    test_frame.to_csv(dest/'test_raw.csv',index=False,float_format='%.12g')
    del core,model,test_prob;gc.collect();torch.cuda.empty_cache()
    if CV_CONFIG['face_oof']:face_oof[val_indices]=fit_face(train,val,dest,fold)
    row['seconds']=time.perf_counter()-start;reports.append(row)
    (OUT/'fold_reports.json').write_text(json.dumps(reports,indent=2))
    emit('fold_complete',family=CV_CONFIG['family'],**row)
assert (seen==1).all() and np.isfinite(oof).all()
np.testing.assert_allclose(oof.sum(1),1,atol=1e-8)
result=metadata[['subject','classname','img','label']].copy();result['fold']=FOLD_IDS;result[CLASSES]=oof
result.to_csv(OUT/'oof_predictions.csv',index=False)
test_frame=sample[['img']].copy();test_frame[CLASSES]=test_sum/5
test_frame.to_csv(OUT/'test_mean_raw.csv',index=False,float_format='%.12g')
if CV_CONFIG['face_oof']:
    assert np.isfinite(face_oof).all() and (face_oof>=0).all() and (face_oof<=1).all()
    face_frame=metadata[['subject','classname','img','label']].copy();face_frame['fold']=FOLD_IDS;face_frame['face_c9']=face_oof
    face_frame.to_csv(OUT/'face_oof_predictions.csv',index=False)
report={'family':CV_CONFIG['family'],'oof_images':len(metadata),'drivers':int(metadata.subject.nunique()),
        'oof_log_loss':float(log_loss(metadata.label,oof,labels=list(range(10)))),
        'oof_accuracy':float(accuracy_score(metadata.label,oof.argmax(1))),
        'test_images':len(sample),'test_models_averaged':5,'folds':reports,
        'every_oof_driver_excluded_from_its_training_model':True,
        'test_labels_used':False,'automatically_submitted':False,'config':CV_CONFIG,
        'elapsed_seconds':time.perf_counter()-STARTED}
(OUT/'cv_report.json').write_text(json.dumps(report,indent=2))
emit('cv_complete',**report)


## 3. 전체 26명 얼굴 보정기 재학습 및 테스트 추론

In [ ]:
from pathlib import Path
import os
INPUT_ROOT=Path(os.environ.get('STATEFARM_INPUT_ROOT','/kaggle/input'))
_default_output='/kaggle/working/vgg_final' if Path('/kaggle/working').is_dir() else './vgg_final_outputs'
WORK_ROOT=Path(os.environ.get('STATEFARM_OUTPUT_ROOT',_default_output))
WORK_ROOT.mkdir(parents=True,exist_ok=True)
import gc,hashlib,json,math,random,time
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import models,transforms
START=time.perf_counter();INPUT=INPUT_ROOT;OUT=WORK_ROOT/'face';OUT.mkdir(parents=True,exist_ok=True)
SEED=42;MAX_EPOCHS=6
assert torch.cuda.is_available(), 'A CUDA GPU is required for training.'
DEVICE=torch.device('cuda:0')
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
normalize=transforms.Normalize([.485,.456,.406],[.229,.224,.225])
augment=transforms.Compose([transforms.Resize((224,224)),
    transforms.RandomAffine(degrees=5,translate=(.04,.04),scale=(.95,1.05),fill=(123,116,103)),
    transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.1),transforms.ToTensor(),normalize])
plain=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),normalize])

def seed_all():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)

def emit(event,**fields):print(json.dumps({'event':event,'arch':'vgg16_face',**fields}),flush=True)

def find_one(pattern):
    found=set()
    for prefix in ('','*/','*/*/'):found.update(INPUT.glob(prefix+pattern))
    if len(found)!=1:raise FileNotFoundError(str(found))
    return next(iter(found))

class Faces(Dataset):
    def __init__(self,frame,training=False,test=False):self.frame=frame.reset_index(drop=True);self.training=training;self.test=test
    def __len__(self):return len(self.frame)
    def __getitem__(self,index):
        row=self.frame.iloc[index]
        path=DATA/'imgs/test'/row['img'] if self.test else DATA/'imgs/train'/row['classname']/row['img']
        with Image.open(path) as original:
            image=original.convert('RGB');w,h=image.size
            image=image.crop((int(.2*w),0,int(.7*w),int(.65*h)))
            image=(augment if self.training else plain)(image)
        return image,index if self.test else int(row['label']==9)

def seed_worker(worker):
    s=torch.initial_seed()%(2**32);random.seed(s);np.random.seed(s)

def loader(frame,training=False,test=False):
    return DataLoader(Faces(frame,training,test),batch_size=64 if test else 32,shuffle=training,
        num_workers=4,pin_memory=True,persistent_workers=False,worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(SEED))

class FaceVGG(nn.Module):
    def __init__(self,features):
        super().__init__();self.features=features;self.pool=nn.AdaptiveAvgPool2d((1,1))
        self.classifier=nn.Sequential(nn.Dropout(.3),nn.Linear(512,2))
    def forward(self,x):return self.classifier(self.pool(self.features(x)).flatten(1))

def make_model():
    seed_all();pre=models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
    core=FaceVGG(pre.features).to(DEVICE);del pre;gc.collect()
    return core,nn.DataParallel(core) if torch.cuda.device_count()>1 else core

def configure(core,model,optimizer,epoch):
    model.train();core.features.requires_grad_(False)
    if epoch==1:rates=[0,1e-3]
    else:
        core.features[17:].requires_grad_(True)
        factor=.2+.8*.5*(1+math.cos(math.pi*(epoch-2)/max(1,MAX_EPOCHS-2)))
        rates=[3e-5*factor,2e-4*factor]
    for group,rate in zip(optimizer.param_groups,rates):group['lr']=rate

def optimizer_for(core):
    return torch.optim.AdamW([{'params':list(core.features[17:].parameters()),'lr':0},
        {'params':core.classifier.parameters(),'lr':1e-3}],weight_decay=1e-4)

def train_epoch(model,data,optimizer,scaler):
    loss_fn=nn.CrossEntropyLoss(label_smoothing=.02);n=correct=0;loss_sum=0.
    for images,labels in data:
        images=images.to(DEVICE,non_blocking=True);labels=labels.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=torch.float16):logits=model(images)
        loss=loss_fn(logits.float(),labels)
        if not torch.isfinite(loss):raise RuntimeError('Nonfinite face loss')
        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
        n+=len(labels);correct+=(logits.argmax(1)==labels).sum().item();loss_sum+=loss.item()*len(labels)
    return loss_sum/n,correct/n

def predict(model,data,test=False):
    model.eval();q=[];indices=[]
    with torch.inference_mode():
        for batch,(images,index) in enumerate(data):
            images=images.to(DEVICE,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16):logits=model(images)
            q.append(logits.double().softmax(1)[:,1].cpu().numpy());indices.append(index.numpy())
            if test and (batch+1)%200==0:emit('test_progress',completed_images=sum(len(x) for x in q))
    return np.concatenate(q),np.concatenate(indices)
metadata_path=find_one('driver_imgs_list.csv');DATA=metadata_path.parent
metadata=pd.read_csv(metadata_path);metadata['label']=metadata.classname.str[1:].astype(int)
assert len(metadata)==22424 and metadata.subject.nunique()==26
binary_full=metadata[metadata.label.isin([0,9])].reset_index(drop=True)
core,model=make_model();optimizer=optimizer_for(core);scaler=torch.amp.GradScaler('cuda')
training=loader(binary_full,True)
for epoch in [1,2]:
    configure(core,model,optimizer,epoch)
    loss,accuracy=train_epoch(model,training,optimizer,scaler)
    emit('full_refit_epoch',epoch=epoch,train_loss=loss,train_accuracy=accuracy)
report={'training_images':len(binary_full),'training_drivers':26,'epochs':2,'temperature_applied':1.,'output_probabilities':'raw'}
torch.save({'model_state_dict':{k:v.detach().cpu() for k,v in core.state_dict().items()},'metadata':report},OUT/'full_face_model.pt')
core.requires_grad_(False);core.zero_grad(set_to_none=True)
del optimizer,scaler,training;gc.collect();torch.cuda.empty_cache()
sample=pd.read_csv(find_one('sample_submission.csv'))
q,indices=predict(model,loader(sample,test=True),True)
np.testing.assert_array_equal(indices,np.arange(len(sample)))
assert np.isfinite(q).all() and (q>=0).all() and (q<=1).all()
pd.DataFrame({'img':sample.img,'face_c9':q}).to_csv(OUT/'test_face_predictions.csv',index=False,float_format='%.12g')
(OUT/'full_training_report.json').write_text(json.dumps(report,indent=2))


## 4. OOF 보정·조합 선택 및 최종 CSV

In [ ]:
from pathlib import Path
import os
INPUT_ROOT=Path(os.environ.get('STATEFARM_INPUT_ROOT','/kaggle/input'))
_default_output='/kaggle/working/vgg_final' if Path('/kaggle/working').is_dir() else './vgg_final_outputs'
WORK_ROOT=Path(os.environ.get('STATEFARM_OUTPUT_ROOT',_default_output))
WORK_ROOT.mkdir(parents=True,exist_ok=True)
"""Select and calibrate fixed VGG CV candidates using labelled OOF data only.

No competition submission. Base OOF excludes each driver's images from its model.
Calibration crossfit is development evaluation, not full nested base-model CV.
"""
from pathlib import Path
import hashlib,json,shutil
import numpy as np
import pandas as pd
from scipy.optimize import minimize,minimize_scalar
from scipy.special import softmax,expit
from sklearn.metrics import log_loss,accuracy_score

OUT=WORK_ROOT/'final';OUT.mkdir(parents=True,exist_ok=True)
ARTIFACT_DIRS={'full':WORK_ROOT/'full','crop':WORK_ROOT/'crop','face':WORK_ROOT/'face'}
SOURCES={key:str(value) for key,value in ARTIFACT_DIRS.items()}
def data_file(name):
    matches=set()
    for prefix in ['', '*/', '*/*/']:matches.update(INPUT_ROOT.glob(prefix+name))
    if len(matches)!=1:raise FileNotFoundError(str((name,sorted(matches))))
    return matches.pop()
CLASSES=[f'c{i}' for i in range(10)]
POLICY={'crop_weights':[0,.5,1],'face_weights':[0,.5,1],
        'global_temperature_bounds':[.5,3],'face_temperature_bounds':[.5,3],
        'class_bias_l2':.01,'bias_adoption_min_gain':.003,'minimum_folds_improved':3,
        'maximum_fold_loss_increase':.03,'candidate_min_gain':.003,
        'driver_bootstrap_lower_bound_positive':True,'seed':20261118,
        'note':'Latest CV-only final recipe. OOF chooses the mixture and calibration; no private notebook dependency or old-submission fallback.'}
(OUT/'calibration_policy.json').write_text(json.dumps(POLICY,indent=2))

def artifact(key,filename):
    path=ARTIFACT_DIRS[key]/filename
    if not path.is_file():raise FileNotFoundError(f'Missing {path}; run the corresponding training stage first.')
    return path

def loss(y,p):return float(-np.log(np.clip(p[np.arange(len(y)),y],1e-15,1)).mean())

def transform(raw,q,params,use_face):
    p=softmax(np.log(np.clip(raw,1e-15,1))/params['temperature']+np.asarray(params['bias']),axis=1)
    if use_face:
        q=np.clip(q,1e-12,1-1e-12)
        q=expit((np.log(q)-np.log1p(-q))/params['face_temperature'])
        mass=p[:,0]+p[:,9]
        q=(1-float(use_face))*p[:,9]/np.clip(mass,1e-15,None)+float(use_face)*q
        p[:,0]=mass*(1-q);p[:,9]=mass*q
    return p

def fit(raw,q,y,mask,use_bias,use_face,fit_face_temperature=True):
    lp=np.log(np.clip(raw[mask],1e-15,1));labels=y[mask];face=q[mask]
    face_t=1.
    if use_face and fit_face_temperature:
        cond=np.isin(labels,[0,9]);qy=np.clip(face[cond],1e-12,1-1e-12)
        zq=np.log(qy)-np.log1p(-qy);by=(labels[cond]==9).astype(float)
        r=minimize_scalar(lambda t:float((np.logaddexp(0,zq/np.exp(t))-by*zq/np.exp(t)).mean()),
                          bounds=(np.log(.5),np.log(3)),method='bounded')
        assert r.success;face_t=float(np.exp(r.x))
    clipped=np.clip(face,1e-12,1-1e-12)
    calibrated_face=expit((np.log(clipped)-np.log1p(-clipped))/face_t)
    def objective(z):
        t=np.exp(z[0]);b=z[1:] if use_bias else np.zeros(10)
        p=softmax(lp/t+b,axis=1);target=np.eye(10)[labels]
        if use_face:
            cond=np.isin(labels,[0,9]);alpha=float(use_face);target[cond]=0
            qy=np.where(labels==9,calibrated_face,1-calibrated_face)
            coef0=alpha*qy+(1-alpha)*(labels==0)
            coef9=alpha*qy+(1-alpha)*(labels==9)
            denominator=np.clip(p[:,0]*coef0+p[:,9]*coef9,1e-15,None)
            target[cond,0]=p[cond,0]*coef0[cond]/denominator[cond]
            target[cond,9]=p[cond,9]*coef9[cond]/denominator[cond]
        g=(p-target)/len(p)
        params={'temperature':float(t),'bias':b.tolist(),'face_temperature':face_t}
        transformed=transform(raw[mask],face,params,use_face)
        value=loss(labels,transformed)
        # Match the clipped log-loss derivative for extremely confident errors.
        active=transformed[np.arange(len(labels)),labels]>1e-15
        g[~active]=0
        gradient=np.array([(g*(-lp/t)).sum()])
        if use_bias:
            value+=.005*np.square(b).sum();gradient=np.r_[gradient,g.sum(0)+.01*b]
        return value,gradient
    init=np.r_[np.log(.6),np.zeros(10)] if use_bias else np.array([np.log(.6)])
    result=minimize(objective,init,jac=True,method='L-BFGS-B',
        bounds=[(np.log(.5),np.log(3))]+([(-3,3)]*10 if use_bias else []),
        options={'ftol':1e-11,'gtol':1e-7,'maxiter':500})
    assert result.success,result.message
    return {'temperature':float(np.exp(result.x[0])),
            'bias':result.x[1:].tolist() if use_bias else [0.]*10,'face_temperature':face_t}

def crossfit(raw,q,y,groups,use_bias,use_face,fit_face_temperature=True):
    pred=np.empty_like(raw);folds=[]
    for held in sorted(set(groups)):
        mask=groups==held
        params=fit(raw,q,y,~mask,use_bias,use_face,fit_face_temperature)
        pred[mask]=transform(raw[mask],q[mask],params,use_face)
        folds.append({'fold':int(held),'n':int(mask.sum()),'log_loss':loss(y[mask],pred[mask]),'parameters':params})
    return pred,{'log_loss':loss(y,pred),'accuracy':float(accuracy_score(y,pred.argmax(1))),'folds':folds}

def driver_interval(reference,pred,y,subjects):
    old=-np.log(np.clip(reference[np.arange(len(y)),y],1e-15,1))
    new=-np.log(np.clip(pred[np.arange(len(y)),y],1e-15,1))
    names=sorted(set(subjects));n=np.array([(subjects==d).sum() for d in names])
    gain=np.array([(old-new)[subjects==d].sum() for d in names])
    ix=np.random.default_rng(20261118).integers(0,len(names),size=(10000,len(names)))
    samples=gain[ix].sum(1)/n[ix].sum(1)
    return {'lower':float(np.quantile(samples,.025)),'upper':float(np.quantile(samples,.975)),
            'drivers_improved':int((gain>0).sum()),'driver_mean_gain':dict(zip(names,(gain/n).tolist()))}

full=pd.read_csv(artifact('full','oof_predictions.csv'))
crop=pd.read_csv(artifact('crop','oof_predictions.csv'))
face=pd.read_csv(artifact('full','face_oof_predictions.csv'))
for c in ['subject','classname','img','label','fold']:
    assert full[c].tolist()==crop[c].tolist()==face[c].tolist()
assert len(full)==22424 and full.subject.nunique()==26 and not full.img.duplicated().any()
assert set(full.fold)==set(range(5)) and full.groupby('subject').fold.nunique().max()==1
official=pd.read_csv(data_file('driver_imgs_list.csv'))
for col in ['subject','classname','img']:assert official[col].tolist()==full[col].tolist()
np.testing.assert_array_equal(full.label.to_numpy(),full.classname.str[1:].astype(int).to_numpy())
for key in ['full','crop']:
    r=json.loads(artifact(key,'cv_report.json').read_text())
    assert r['every_oof_driver_excluded_from_its_training_model']
    assert r['test_labels_used'] is False and r['automatically_submitted'] is False
    for f in r['folds']:
        assert not set(f['fit_drivers'])&set(f['validation_drivers'])
        assert f['epochs_fixed_before_training']==5 and not f['validation_used_for_checkpoint_selection']
        assert set(full.loc[full.fold==f['fold'],'subject'])==set(f['validation_drivers'])
Y=full.label.to_numpy();GROUPS=full.fold.to_numpy();SUBJECTS=full.subject.to_numpy()
q=face.face_c9.to_numpy();P=full[CLASSES].to_numpy();C=crop[CLASSES].to_numpy()
assert np.isfinite(q).all() and (q>=0).all() and (q<=1).all()
for p in [P,C]:
    assert np.isfinite(p).all() and (p>=0).all() and (p<=1).all()
    np.testing.assert_allclose(p.sum(1),1,atol=1e-8)

candidates=[];selected_families=[]
for weight in POLICY['crop_weights']:
    raw=(1-weight)*P+weight*C
    for use_face in POLICY['face_weights']:
        options=[]
        for use_bias in [False,True]:
            pred,report=crossfit(raw,q,Y,GROUPS,use_bias,use_face)
            row={'crop_weight':weight,'face':use_face,'class_bias':use_bias,**report,'pred':pred}
            options.append(row);candidates.append(row)
            print(json.dumps({k:v for k,v in row.items() if k not in ['pred','folds']}),flush=True)
        scalar,vector=options
        gains=[a['log_loss']-b['log_loss'] for a,b in zip(scalar['folds'],vector['folds'])]
        adopt=(vector['log_loss']<scalar['log_loss']-.003 and sum(g>0 for g in gains)>=3 and min(gains)>=-.03)
        selected_families.append(vector if adopt else scalar)
baseline=next(r for r in selected_families if r['crop_weight']==0 and r['face']==1)
eligible=[baseline]
for r in selected_families:
    interval=driver_interval(baseline['pred'],r['pred'],Y,SUBJECTS)
    r['driver_interval_vs_full_face']=interval
    gains=[a['log_loss']-b['log_loss'] for a,b in zip(baseline['folds'],r['folds'])]
    if r['log_loss']<baseline['log_loss']-.003 and sum(g>0 for g in gains)>=3 and min(gains)>=-.03 and interval['lower']>0:
        eligible.append(r)
choice=min(eligible,key=lambda r:(r['log_loss'],int(r['crop_weight']==.5),r['face']))
chosen_raw=(1-choice['crop_weight'])*P+choice['crop_weight']*C
chosen_params=fit(chosen_raw,q,Y,np.ones(len(Y),dtype=bool),choice['class_bias'],choice['face'])
chosen_recipe={k:v for k,v in choice.items() if k!='pred'}
chosen_recipe['full_fit_parameters']=chosen_params
validation=full[['subject','classname','img','label','fold']].copy();validation[CLASSES]=choice['pred']
validation.to_csv(OUT/'selected_oof_predictions.csv',index=False)
pd.DataFrame([{k:r[k] for k in ['crop_weight','face','class_bias','log_loss','accuracy']} for r in candidates]).to_csv(OUT/'candidate_scores.csv',index=False)


(OUT/'selected_recipe.json').write_text(json.dumps(chosen_recipe,indent=2))
full_fit={'recipe':chosen_recipe,'sources':SOURCES,'parameters_frozen_before_test_read':True}
(OUT/'decision_frozen_before_test_read.json').write_text(json.dumps(full_fit,indent=2))
ta=pd.read_csv(artifact('full','test_mean_raw.csv'));tc=pd.read_csv(artifact('crop','test_mean_raw.csv'))
tf=pd.read_csv(artifact('face','test_face_predictions.csv'))
official_sample=pd.read_csv(data_file('sample_submission.csv'))
assert ta.img.tolist()==tc.img.tolist()==tf.img.tolist()==official_sample.img.tolist()
assert len(ta)==79726
raw=(1-choice['crop_weight'])*ta[CLASSES].to_numpy()+choice['crop_weight']*tc[CLASSES].to_numpy()
prob=transform(raw,tf.face_c9.to_numpy(),chosen_params,choice['face'])
result=ta[['img']].copy();result[CLASSES]=prob
result.to_csv(OUT/'submission.csv',index=False,float_format='%.12g')
saved=pd.read_csv(OUT/'submission.csv');p=saved[CLASSES].to_numpy()
assert saved.columns.tolist()==['img']+CLASSES and len(saved)==79726 and not saved.img.duplicated().any()
assert np.isfinite(p).all() and (p>=0).all() and (p<=1).all()
np.testing.assert_allclose(p.sum(1),1,atol=1e-9)
report={'rows':len(saved),'sha256':hashlib.sha256((OUT/'submission.csv').read_bytes()).hexdigest(),
        'oof_log_loss':choice['log_loss'],'oof_accuracy':choice['accuracy'],
        'recipe':chosen_recipe,'test_labels_used':False,'automatically_submitted':False,
        'kaggle_public_score':None,'kaggle_private_score':None}
(OUT/'submission_validation.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report),flush=True)
